In [9]:
import os
import numpy as np
import pandas as pd
import folium
import rasterio
import osmnx as ox
import geopandas as gpd
import networkx as nx

In [ ]:
from shapely.geometry import Point

# --- Configuration ---
DATA_DIR = '/Users/gunottammaini/Desktop/PROJECTS/PBL 4th Sem/STATS AND AI/project'
DAMAGE_MAP_PATH = os.path.join(DATA_DIR, 'ARIA_DamageProxyMap_v0.5u_CSKd_20150430.tif')
UNOSAT_SHP_PATH = os.path.join(DATA_DIR, 'All_Damage_Nepal_April_28th_2015.shp')
OSM_DATA_PATH = os.path.join(DATA_DIR, 'kathmandu_buildings.geojson')

# --- Part 1: Data Loading and Processing ---
def load_damage_data():
    """Load and process damage assessment data from multiple sources."""
    damage_scores = {}

    # Load NASA damage proxy map if available
    if os.path.exists(DAMAGE_MAP_PATH):
        try:
            with rasterio.open(DAMAGE_MAP_PATH) as src:
                damage_data = src.read(1)
                # Normalize damage scores (example - adjust based on your data)
                min_val = np.nanmin(damage_data)
                max_val = np.nanmax(damage_data)
                if max_val > min_val:
                    normalized_data = (damage_data - min_val) / (max_val - min_val)
                    # Mask NaN values to avoid issues in calculations
                    damage_scores['nasa'] = np.nan_to_num(normalized_data, nan=0.5)
                else:
                    print("Warning: NASA damage data has no variance, skipping normalization.")
        except rasterio.RasterioIOError as e:
            print(f"Error loading NASA damage data: {e}")

    # Load UNOSAT damage assessments
    if os.path.exists(UNOSAT_SHP_PATH):
        try:
            unosat_gdf = gpd.read_file(UNOSAT_SHP_PATH).to_crs("EPSG:4326")
            # Convert UNOSAT damage to numerical scores
            damage_mapping = {'AFFECTED': 1.0, 'POSSIBLY AFFECTED': 0.7, 'NO VISIBLE DAMAGE': 0.1}
            unosat_gdf['damage_score'] = unosat_gdf['Damage'].map(damage_mapping).fillna(0.5)
            damage_scores['unosat'] = unosat_gdf
        except fiona.errors.DriverError as e:
            print(f"Error loading UNOSAT data: {e}")

    return damage_scores

def load_osm_data():
    """Load OSM building and infrastructure data for Kathmandu."""
    if os.path.exists(OSM_DATA_PATH):
        print("Loading cached OSM data...")
        gdf = gpd.read_file(OSM_DATA_PATH)
    else:
        print("Downloading OSM data for Kathmandu...")
        tags = {
            'building': True,
            'amenity': ['hospital', 'clinic', 'school', 'government'],
            'emergency': ['fire_station', 'police']
        }
        try:
            gdf = ox.features_from_place('Kathmandu, Nepal', tags=tags)
            gdf = gdf.to_crs("EPSG:4326")
            gdf.to_file(OSM_DATA_PATH, driver='GeoJSON')
        except Exception as e:
            print(f"Error downloading OSM data: {e}")
            return None
    return gdf

def process_osm_data(gdf):
    """Process the OSM GeoDataFrame to classify infrastructure types and add essential columns."""
    if gdf is None:
        return None

    gdf['infra_type'] = 'residential'  # Default

    # Convert building levels to numeric safely
    gdf['building:levels'] = pd.to_numeric(gdf['building:levels'], errors='coerce')

    # Healthcare facilities
    gdf.loc[gdf['amenity'].isin(['hospital', 'clinic']), 'infra_type'] = 'healthcare'

    # Schools
    gdf.loc[gdf['amenity'] == 'school', 'infra_type'] = 'school'

    # Government buildings
    gdf.loc[gdf['amenity'] == 'government', 'infra_type'] = 'government'

    # Emergency services
    gdf.loc[gdf['emergency'].notna(), 'infra_type'] = 'emergency'

    # Multi-story buildings (3+ levels)
    gdf.loc[gdf['building:levels'] >= 3, 'infra_type'] = 'multi_story'

    # Add essential columns with fallbacks
    gdf['building:material'] = gdf.get('building:material', 'RC')               

    return gdf

# --- Part 2: Priority Calculation and Visualization ---
def calculate_priority_scores(gdf, damage_scores):
    """Calculate meaningful priority scores based on multiple factors."""
    if gdf is None:
        return None

    # Material vulnerability weights
    '''
    material_weights = {
        'Mud': 0.9,
        'Timber': 0.7,
        'RC': 0.3,
        'Brick': 0.5,
        'Steel': 0.2
    }
    '''

    # Infrastructure type weights
    type_weights = {
        'healthcare': 3.0,
        'emergency': 2.5,
        'government': 2.0,
        'school': 2.0,
        'multi_story': 1.5,
        'residential': 1.0
    }

    # Calculate base scores
    gdf['material_score'] = gdf['building:material'].map(material_weights).fillna(0.5)
    gdf['type_weight'] = gdf['infra_type'].map(type_weights).fillna(1.0)

    # Incorporate damage data
    if 'unosat' in damage_scores and not damage_scores['unosat'].empty and not gdf.empty:
        try:
            gdf = gpd.sjoin(gdf, damage_scores['unosat'][['geometry', 'damage_score']],
                            how='left', predicate='intersects')
            gdf['damage_score'] = gdf['damage_score'].fillna(0.5)
            gdf.drop(columns=['index_right'], inplace=True) # Clean up after join
        except Exception as e:
            print(f"Error during spatial join with UNOSAT data: {e}")
            gdf['damage_score'] = 0.5
    else:
        gdf['damage_score'] = 0.5  # Default if no UNOSAT data or gdf is empty

    # Calculate area and normalize
    if not gdf.empty:
        gdf['plinth_area_sq_ft'] = gdf.geometry.area * 10.764  # Convert m² to ft²
        min_area = gdf['plinth_area_sq_ft'].min()
        max_area = gdf['plinth_area_sq_ft'].max()
        if max_area > min_area:
            gdf['area_norm'] = (gdf['plinth_area_sq_ft'] - min_area) / (max_area - min_area)
        else:
            gdf['area_norm'] = 0.5 # Default if no area variance
    else:
        gdf['area_norm'] = 0.5

    # Final priority calculation
    gdf['priority_score'] = (
        (gdf['damage_score'] * 0.4) +
        (gdf['material_score'] * 0.3) +
        (gdf['type_weight'] * 0.2) +
        (gdf['area_norm'] * 0.1))

    # Scale to 1-10 range for interpretability
    gdf['priority_score'] = 1 + (gdf['priority_score'] * 9)

    return gdf

def create_priority_map(gdf):
    """Create and save a Folium map visualizing the priority scores."""
    if gdf is None or gdf.empty:
        print("No data to create priority map.")
        return

    print("Creating priority map...")
    m = folium.Map(location=[27.7172, 85.3240], zoom_start=12)

    # Color mapping
    colors = {
        'healthcare': 'red',
        'emergency': 'orange',
        'school': 'purple',
        'government': 'darkblue',
        'multi_story': 'blue',
        'residential': 'gray'
    }
    gdf['color'] = gdf['infra_type'].map(colors)

    # Add markers
    for _, row in gdf.iterrows():
        if not pd.isna(row.geometry.centroid.y):
            folium.CircleMarker(
                location=[row.geometry.centroid.y, row.geometry.centroid.x],
                radius=row['priority_score'] * 0.5,  # Scale for visibility
                color=row['color'],
                fill=True,
                fill_opacity=0.7,
                popup=folium.Popup(
                    f"<b>Type:</b> {row['infra_type']}<br>"
                    f"<b>Priority:</b> {row['priority_score']:.1f}/10<br>"
                    f"<b>Damage:</b> {row['damage_score']:.1f}<br>"
                    f"<b>Material:</b> {row['building:material']}<br>"
                    f"<b>Levels:</b> {row.get('building:levels', 'N/A')}",
                    max_width=300
                )
            ).add_to(m)

    # Save and display
    m.save('priority_map.html')
    print("✅ Priority map saved with meaningful scores")
    print("\nPriority score distribution:")
    print(gdf['priority_score'].describe())
    print("\nDamage score distribution:")
    print(gdf['damage_score'].describe())
    
if __name__ == "__main__":
    # Part 1: Load and Process Data
    damage_data = load_damage_data()
    osm_gdf = load_osm_data()
    processed_gdf = process_osm_data(osm_gdf)

    # Part 2: Calculate Priorities and Visualize
    if processed_gdf is not None:
        prioritized_gdf = calculate_priority_scores(processed_gdf.copy(), damage_data) # Use a copy to avoid modifying the original
        create_priority_map(prioritized_gdf)

/var/folders/gk/6047c5r144d9vbvbr0cvckrr0000gn/T/ipykernel_75153/2426220752.py:146: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['plinth_area_sq_ft'] = gdf.geometry.area * 10.764  # Convert m² to ft²


Creating priority map...


In [6]:
prioritized_gdf

geometry  \
element id                                                              
node    313140391                           POINT (85.32846 27.71433)   
        575231600                           POINT (85.30179 27.72018)   
        845503449                           POINT (85.30892 27.73141)   
        943246318                           POINT (85.31124 27.71794)   
        1497172311                          POINT (85.29132 27.71144)   
...                                                               ...   
way     1384267963  POLYGON ((85.35722 27.68302, 85.35714 27.68291...   
        1384267965  POLYGON ((85.35683 27.68279, 85.3569 27.68275,...   
        1384269753  POLYGON ((85.35447 27.67693, 85.35451 27.67702...   
        1384269754  POLYGON ((85.3542 27.67697, 85.35426 27.67696,...   
        1384269755  POLYGON ((85.3548 27.67627, 85.355 27.67621, 8...   

                       amenity building           int_name  \
element id                                                   
node    313140391   restaurant      yes  Moskva Restaurant   
        575231600       school      NaN                NaN   
        845503449          NaN   office                NaN   
        943246318     hospital      NaN                NaN   
        1497172311    hospital      NaN                NaN   
...                        ...      ...                ...   
way     1384267963         NaN      yes                NaN   
        1384267965         NaN      yes                NaN   
        1384269753         NaN      yes                NaN   
        1384269754         NaN      yes                NaN   
        1384269755         NaN      yes                NaN   

                                       name                     name:en  \
element id                                                                
node    313140391                   Wunjala                         NaN   
        575231600    अक्सफर्ड बोर्डिङ स्कुल      Oxford Boarding School   
        845503449             Nepal Telecom                         NaN   
        943246318   मनमोहन मेमोरियल अस्पताल  Manmohan Memorial Hospital   
        1497172311            सैनिक अस्पताल           Military Hospital   
...                                     ...                         ...   
way     1384267963                      NaN                         NaN   
        1384267965                      NaN                         NaN   
        1384269753                      NaN                         NaN   
        1384269754                      NaN                         NaN   
        1384269755                      NaN                         NaN   

                                    name:ne             office  \
element id                                                       
node    313140391                       NaN                NaN   
        575231600    अक्सफर्ड बोर्डिङ स्कुल                NaN   
        845503449                       NaN  telecommunication   
        943246318   मनमोहन मेमोरियल अस्पताल                NaN   
        1497172311            सैनिक अस्पताल                NaN   
...                                     ...                ...   
way     1384267963                      NaN                NaN   
        1384267965                      NaN                NaN   
        1384269753                      NaN                NaN   
        1384269754                      NaN                NaN   
        1384269755                      NaN                NaN   

                   toilets:wheelchair wheelchair  ... district province  \
element id                                        ...                     
node    313140391                 NaN        NaN  ...      NaN      NaN   
        575231600                 NaN        NaN  ...      NaN      NaN   
        845503449                 NaN        NaN  ...      NaN      NaN   
        943246318                 NaN        NaN  ...      NaN      NaN   
        1497172311 

In [8]:
# In your main() function, after calculating priorities:
prioritized_gdf.to_file('prioritized_buildings.geojson', driver='GeoJSON')
print("✅ Prioritized buildings data saved to prioritized_buildings.geojson")

✅ Prioritized buildings data saved to prioritized_buildings.geojson


In [11]:
import osmnx as ox

place_name = "Kathmandu, Nepal"

try:
    # Download the road network as a graph
    roads = ox.graph_from_place(place_name, network_type='drive')

    # Save the graph to a file (optional, but good for persistence)
    ox.save_graphml(roads, "kathmandu_roads.graphml")
    print("✅ Kathmandu road network downloaded and saved as kathmandu_roads.graphml")

except Exception as e:
    print(f"⚠️ Error downloading road network: {e}")
    roads = None

✅ Kathmandu road network downloaded and saved as kathmandu_roads.graphml


In [36]:
import networkx as nx
from pyproj import CRS

try:
    roads = ox.load_graphml("kathmandu_roads.graphml")
    print("✅ Loaded Kathmandu road network.")
except FileNotFoundError:
    print("⚠️ Error: kathmandu_roads.graphml not found.")
    roads = None

if prioritized_gdf is not None and roads is not None:
    try:
        # Use updated GeoPandas method
        centroid = prioritized_gdf.union_all().centroid
        centroid_lat, centroid_lon = centroid.y, centroid.x

        # Use pyproj instead of deprecated OSMnx function
        utm_zone = int((centroid_lon + 180) / 6) + 1
        hemisphere = 'north' if centroid_lat >= 0 else 'south'
        target_crs = CRS.from_proj4(f"+proj=utm +zone={utm_zone} +{hemisphere} +datum=WGS84 +units=m +no_defs")

        # Project both datasets
        prioritized_gdf_proj = prioritized_gdf.to_crs(target_crs)
        roads_proj = ox.project_graph(roads, to_crs=target_crs)

        print(f"✅ Data projected to {target_crs}")
    except Exception as e:
        print(f"⚠️ Error projecting data: {str(e)}")
        print(f"Error details: {repr(e)}")
        roads_proj = None
        prioritized_gdf_proj = None

    if roads_proj is not None and prioritized_gdf_proj is not None:
        dependency_graph = nx.Graph()

        for u, v, data in roads_proj.edges(data=True):
            dependency_graph.add_edge(f"road_{u}", f"road_{v}", **data)
        for n, data in roads_proj.nodes(data=True):
            dependency_graph.add_node(f"road_{n}", type="road", **data)

        from shapely.geometry import Point

        for index, row in prioritized_gdf_proj.iterrows():
            if row.geometry.geom_type == 'Point':
                building_id = row.get('osmid', f"building_{index}")
                
                # Avoid duplicate 'type' key
                row_dict = row.to_dict()
                row_dict.pop("type", None)
                
                # Add building node to the graph
                dependency_graph.add_node(building_id, type="building", **row_dict)
        
                # Find nearest road node
                nearest_road_node = ox.distance.nearest_nodes(
                    roads_proj, 
                    X=[row.geometry.x], 
                    Y=[row.geometry.y]
                )[0]
        
                if nearest_road_node is not None:
                    # Create Point for road node
                    road_point = Point(
                        roads_proj.nodes[nearest_road_node]['x'],
                        roads_proj.nodes[nearest_road_node]['y']
                    )
                    distance = row.geometry.distance(road_point)
        
                    # Add edge between building and nearest road node
                    dependency_graph.add_edge(
                        building_id, 
                        f"road_{nearest_road_node}", 
                        relation="access",
                        distance=distance
                    )
                    print(f"🔗 Connected building {building_id} to road node {nearest_road_node}")
                else:
                    print(f"⚠️ Could not find nearest road for building {building_id}")

        import numpy as np

        # Convert all node attributes to string or primitive types
        for node, data in dependency_graph.nodes(data=True):
            cleaned_data = {}
            for k, v in data.items():
                if isinstance(v, (np.generic, np.ndarray)):
                    v = v.item() if hasattr(v, 'item') else str(v)
                elif isinstance(v, (dict, list, set, tuple)):
                    v = str(v)
                cleaned_data[k] = str(v)  # GML expects string values
            dependency_graph.nodes[node].clear()
            dependency_graph.nodes[node].update(cleaned_data)
        
        # Convert all edge attributes similarly
        for u, v, data in dependency_graph.edges(data=True):
            cleaned_data = {}
            for k, v_ in data.items():
                if isinstance(v_, (np.generic, np.ndarray)):
                    v_ = v_.item() if hasattr(v_, 'item') else str(v_)
                elif isinstance(v_, (dict, list, set, tuple)):
                    v_ = str(v_)
                cleaned_data[k] = str(v_)
            dependency_graph.edges[u, v].clear()
            dependency_graph.edges[u, v].update(cleaned_data)
        # Sanitize attribute keys to remove ':' characters (not allowed in GML)
        def sanitize_attrs(attrs):
            new_attrs = {}
            for k, v in attrs.items():
                new_key = k.replace(":", "_")
                new_attrs[new_key] = v
            return new_attrs
        
        for node, data in dependency_graph.nodes(data=True):
            sanitized_data = sanitize_attrs(data)
            dependency_graph.nodes[node].clear()
            dependency_graph.nodes[node].update(sanitized_data)
        
        for u, v, data in dependency_graph.edges(data=True):
            sanitized_data = sanitize_attrs(data)
            dependency_graph.edges[u, v].clear()
            dependency_graph.edges[u, v].update(sanitized_data)


        import re
        
        def sanitize_key(key):
            # Replace any character not alphanumeric or underscore with underscore
            return re.sub(r'[^a-zA-Z0-9_]', '_', key)
        
        # For nodes
        for node, data in dependency_graph.nodes(data=True):
            cleaned_data = {}
            for k, v in data.items():
                new_key = sanitize_key(k)
                if isinstance(v, (np.generic, np.ndarray)):
                    v = v.item() if hasattr(v, 'item') else str(v)
                elif isinstance(v, (dict, list, set, tuple)):
                    v = str(v)
                cleaned_data[new_key] = str(v)
            dependency_graph.nodes[node].clear()
            dependency_graph.nodes[node].update(cleaned_data)
        
        # For edges
        for u, v, data in dependency_graph.edges(data=True):
            cleaned_data = {}
            for k, v_ in data.items():
                new_key = sanitize_key(k)
                if isinstance(v_, (np.generic, np.ndarray)):
                    v_ = v_.item() if hasattr(v_, 'item') else str(v_)
                elif isinstance(v_, (dict, list, set, tuple)):
                    v_ = str(v_)
                cleaned_data[new_key] = str(v_)
            dependency_graph.edges[u, v].clear()
            dependency_graph.edges[u, v].update(cleaned_data)

        # Now save
        try:
            nx.write_gml(dependency_graph, "dependency_graph_roads_with_priority.gml")
            print("✅ Dependency graph saved as dependency_graph_roads_with_priority.gml")
        except Exception as e:
            print(f"⚠️ Error saving GML file: {e}")

    else:
        print("⚠️ Unable to proceed with dependency graph creation due to projection errors.")
else:
    print("⚠️ Prioritized building data or road network not loaded correctly.")


✅ Loaded Kathmandu road network.
✅ Data projected to +proj=utm +zone=45 +north +datum=WGS84 +units=m +no_defs +type=crs
🔗 Connected building building_('node', 313140391) to road node 1279198216
🔗 Connected building building_('node', 575231600) to road node 1275284561
🔗 Connected building building_('node', 845503449) to road node 1980601139
🔗 Connected building building_('node', 943246318) to road node 8523593236
🔗 Connected building building_('node', 1497172311) to road node 1832337550
🔗 Connected building building_('node', 1741695868) to road node 1741701789
🔗 Connected building building_('node', 1892710734) to road node 1281234039
🔗 Connected building building_('node', 1892728518) to road node 10918103015
🔗 Connected building building_('node', 1892796767) to road node 1274183665
🔗 Connected building building_('node', 1894151516) to road node 1965933572
🔗 Connected building building_('node', 1894209925) to road node 2486251825
🔗 Connected building building_('node', 1894214363) to road

In [50]:
import osmnx as ox
import geopandas as gpd

def get_features(place, tags):
    try:
        # Use the correct function for the current version of osmnx
        return ox.features_from_place(place, tags=tags)
    except Exception as e:
        print(f"⚠️ Error in get_features: {e}")
        return None

# Main function to download and process infrastructure data
def download_infrastructure_data(place_name, target_crs):
    """
    Downloads and processes infrastructure data for a given place.
    
    Args:
        place_name (str): Name of the place to download data for (e.g., 'Kathmandu, Nepal')
        target_crs: The CRS to project the data to (must be the same as your prioritized_gdf_proj)
    
    Returns:
        tuple: (power_lines_proj, water_proj) - projected GeoDataFrames for power and water infrastructure
    """
    # Initialize variables
    power_lines_proj = None
    water_proj = None
    
    # Download and process power lines
    try:
        power_lines = get_features(place_name, tags={'power': 'line'})
        if power_lines is not None and not power_lines.empty:
            power_lines_proj = power_lines.to_crs(target_crs)
            print("✅ Power lines downloaded and projected.")
        else:
            print("⚠️ No power lines data available.")
    except Exception as e:
        print(f"⚠️ Error downloading/projecting power lines: {e}")

    # Download and process water infrastructure
    try:
        water_networks = get_features(place_name, tags={'waterway': 'pipe'})
        water_towers = get_features(place_name, tags={'amenity': 'water_tower', 'building': 'yes'})

        water = None
        if water_networks is not None and not water_networks.empty:
            water = water_networks.copy()
        if water_towers is not None and not water_towers.empty:
            if water is not None:
                water = gpd.pd.concat([water, water_towers], ignore_index=True)
            else:
                water = water_towers.copy()

        if water is not None and not water.empty:
            water_proj = water.to_crs(target_crs)
            print("✅ Water infrastructure downloaded and projected.")
        else:
            print("⚠️ No water infrastructure data available.")
    except Exception as e:
        print(f"⚠️ Error downloading/projecting water infrastructure: {e}")

    return power_lines_proj, water_proj

# Example usage:
# You need to define your target CRS (this should be the same as your prioritized_gdf_proj.crs)
# target_crs = prioritized_gdf_proj.crs  # Uncomment and replace with your actual CRS

# Then call the function:
# power_lines_proj, water_proj = download_infrastructure_data('Kathmandu, Nepal', target_crs)

In [51]:
# Define your target CRS (this should match your prioritized_gdf_proj's CRS)
target_crs = "EPSG:32645"  # Example UTM zone for Nepal - replace with your actual CRS

# Download the data
power_data, water_data = download_infrastructure_data('Kathmandu, Nepal', target_crs)

✅ Power lines downloaded and projected.
⚠️ Error in get_features: No matching features. Check query location, tags, and log.
✅ Water infrastructure downloaded and projected.


In [78]:
import osmnx as ox
import geopandas as gpd
import networkx as nx
from shapely.ops import nearest_points

# Load the dependency graph with road connections

try:
    dependency_graph = nx.read_gml("dependency_graph_roads_with_priority.gml")
    print("✅ Loaded dependency graph with road connections.")
except FileNotFoundError:
    print("⚠️ Error: dependency_graph_roads_with_priority.gml not found.")
    dependency_graph = None

# Load the prioritized buildings data (projected)
try:
    prioritized_gdf_proj = gpd.read_file('prioritized_buildings.geojson')
    # Use ox.projection.project_gdf to get projected crs and reproject
    projected = ox.projection.project_gdf(prioritized_gdf_proj)
    prioritized_gdf_proj = projected  # already projected
    print("✅ Loaded and projected prioritized buildings data.")
except FileNotFoundError:
    print("⚠️ Error: prioritized_buildings.geojson not found.")
    prioritized_gdf_proj = None
except Exception as e:
    print(f"⚠️ Error loading/projecting prioritized buildings: {e}")
    prioritized_gdf_proj = None

power_lines_proj = power_data
water_proj = water_data


if dependency_graph is not None and prioritized_gdf_proj is not None:
    # Connect buildings to power network
    if power_lines_proj is not None:
        print("\nConnecting buildings to power network...")
        for index, building in prioritized_gdf_proj.iterrows():
            if building.geometry.geom_type == 'Point':
                building_id = building.get('osmid', f"building_{index}")
                nearest_power_geom = None
                min_distance_power = float('inf')
                nearest_power_id = None

                for _, power_line in power_lines_proj.iterrows():
                    if power_line.geometry is not None:
                        closest_point_building, closest_point_line = nearest_points(building.geometry, power_line.geometry)
                        distance = closest_point_building.distance(closest_point_line)
                        if distance < min_distance_power:
                            min_distance_power = distance
                            nearest_power_geom = power_line.geometry
                            nearest_power_id = power_line.get('osm_id', f"power_line_{_}")

                if nearest_power_geom is not None and min_distance_power < 200: # Adjust threshold as needed
                    power_line_node_id = f"power_line_{nearest_power_id}"
                    if power_line_node_id not in dependency_graph:
                        dependency_graph.add_node(power_line_node_id, type="power_line", geometry=nearest_power_geom)
                    dependency_graph.add_edge(building_id, power_line_node_id, relation="power_supply", distance=min_distance_power)
                    print(f"🔗 Connected building {building_id} to power line {nearest_power_id} (distance: {min_distance_power:.2f})")

        from shapely.ops import nearest_points
        from shapely.geometry import Point
        import pandas as pd
        
        # Build spatial index once
        water_sindex = water_proj.sindex
        
        # Convert water geometries to list (to avoid repeated Series access)
        water_geoms = water_proj.geometry.values
        water_features = water_proj
        
        print("\nOptimized: Connecting buildings to water network...")
        for index, building in prioritized_gdf_proj.iterrows():
            if building.geometry.geom_type == 'Point':
                building_id = building.get('osmid', f"building_{index}")
                building_geom = building.geometry
        
                # Use spatial index to get nearby geometries (bounding box buffer)
                possible_matches_index = list(water_sindex.intersection(building_geom.bounds))
                if not possible_matches_index:
                    continue
        
                # Filter possible candidates
                candidates = water_features.iloc[possible_matches_index]
        
                # Find nearest using exact geometry
                min_distance = float('inf')
                nearest_geom = None
                nearest_water_type = None
                nearest_water_id = None
        
                for _, feature in candidates.iterrows():
                    if feature.geometry is None:
                        continue
                    p1, p2 = nearest_points(building_geom, feature.geometry)
                    distance = p1.distance(p2)
                    if distance < min_distance:
                        min_distance = distance
                        nearest_geom = feature.geometry
                        nearest_water_type = feature.get('waterway', feature.get('amenity'))
                        nearest_water_id = feature.get('osm_id', f"water_feature_{_}")
        
                if nearest_geom is not None and min_distance < 100:
                    water_node_id = f"water_{nearest_water_id}"
                    if water_node_id not in dependency_graph:
                        dependency_graph.add_node(water_node_id, type=nearest_water_type, geometry=nearest_geom)
                    dependency_graph.add_edge(building_id, water_node_id, relation="water_supply", distance=min_distance)
                    print(f"🔗 Connected building {building_id} to water {nearest_water_type} {nearest_water_id} (distance: {min_distance:.2f})")

        # Create a shallow copy of the graph
        G_serializable = nx.DiGraph()
        
        # Copy nodes with serializable attributes
        for node, attrs in dependency_graph.nodes(data=True):
            attrs_copy = {k: (str(v) if k == 'geometry' else v) for k, v in attrs.items() if k != 'geometry' or isinstance(v, (str, int, float, bool))}
            G_serializable.add_node(node, **attrs_copy)
        
        # Copy edges with serializable attributes
        for u, v, attrs in dependency_graph.edges(data=True):
            attrs_copy = {k: (str(v) if k == 'geometry' else v) for k, v in attrs.items() if k != 'geometry' or isinstance(v, (str, int, float, bool))}
            G_serializable.add_edge(u, v, **attrs_copy)
        
        # Now write
        nx.write_gml(G_serializable, "dependency_graph_full.gml")
        print("✅ Full dependency graph saved as dependency_graph_full.gml")

else:
    print("⚠️ Could not load dependency graph or prioritized building data for extension.")

✅ Loaded dependency graph with road connections.


/Users/gunottammaini/Desktop/PROJECTS/PBL 4th Sem/STATS AND AI/project/venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: Several features with id = 2694829 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


✅ Loaded and projected prioritized buildings data.

Connecting buildings to power network...
🔗 Connected building building_134 to power line power_line_('way', 309114167) (distance: 122.06)
🔗 Connected building building_176 to power line power_line_('way', 756860368) (distance: 75.26)
🔗 Connected building building_225 to power line power_line_('way', 341450813) (distance: 102.46)
🔗 Connected building building_285 to power line power_line_('way', 309114164) (distance: 91.14)
🔗 Connected building building_364 to power line power_line_('way', 309114167) (distance: 165.93)
🔗 Connected building building_412 to power line power_line_('way', 341450813) (distance: 1.37)
🔗 Connected building building_505 to power line power_line_('way', 309114167) (distance: 169.01)

Optimized: Connecting buildings to water network...
🔗 Connected building building_0 to water restaurant water_feature_('node', 313140391) (distance: 0.00)
🔗 Connected building building_1 to water nan water_feature_('way', 651990988

In [ ]:
# Step 6

In [34]:
import networkx as nx
import random
import pandas as pd
from pyvis.network import Network

class DisasterRecoverySystem:
    def __init__(self, graph):
        self.graph = graph
        self.priority_list = []
        self.betweenness = {}

    def _calculate_accessibility_score(self, node):
        count = 0
        for neighbor in self.graph.neighbors(node):
            edge_data = self.graph.get_edge_data(node, neighbor)
            condition = edge_data.get('condition', 'good')
            if 'road' in edge_data.get('type', '') and condition == 'good':
                count += 1
        return min(count / 5.0, 1.0)

    def analyze_impacts(self):
        print("\n=== Critical Infrastructure Analysis ===")

        critical_nodes = [n for n, data in self.graph.nodes(data=True)
                          if data.get('type') in ['hospital', 'power_line', 'water']]

        subgraph = self.graph.subgraph(critical_nodes)
        self.betweenness = nx.betweenness_centrality(subgraph)

        def type_weight(node_type):
            return {'hospital': 1.0, 'power_line': 0.7, 'water': 0.5}.get(node_type, 0.3)

        scored_nodes = []
        for node in critical_nodes:
            b_score = self.betweenness.get(node, 0)
            a_score = self._calculate_accessibility_score(node)
            t_score = type_weight(self.graph.nodes[node].get('type'))
            combined_score = 0.5 * b_score + 0.3 * a_score + 0.2 * t_score
            scored_nodes.append((node, combined_score))

        top_nodes = sorted(scored_nodes, key=lambda x: x[1], reverse=True)[:5]

        print("\nTop 5 Critical Nodes (Combined Score):")
        for node, score in top_nodes:
            node_type = self.graph.nodes[node].get('type', 'unknown')
            print(f"- {node} ({node_type}, Score: {score:.2f})")
            self.priority_list.append(node)

        print("\nWorst-case Failure Impacts:")
        for node in self.priority_list[:3]:
            affected = list(nx.single_source_shortest_path_length(self.graph, node, cutoff=2))
            print(f"If {node} fails, it affects {len(affected) - 1} nodes within 2 hops")

    def generate_recommendations(self):
        print("\n=== Recovery Recommendations ===")
        recommendations = []

        for node in self.priority_list:
            node_data = self.graph.nodes[node]
            rec = {
                'node_id': node,
                'type': node_data.get('type'),
                'action': self._get_action(node_data),
                'urgency': self.betweenness.get(node, 0)
            }
            recommendations.append(rec)

        return pd.DataFrame(recommendations)

    def _get_action(self, node_data):
        node_type = node_data.get('type')
        if node_type == 'power_line':
            return "Immediate repair with backup generator (Est. 6 hrs)"
        elif node_type == 'hospital':
            return "Priority crew allocation (Est. 8 hrs)"
        elif node_type == 'water':
            return "Standard protocol, ensure water trucks (Est. 10 hrs)"
        else:
            return "Standard repair protocol (Est. 12 hrs)"

    def visualize_interactive(self, num_critical_nodes=20, output_file="critical_network.html"):
        net = Network(notebook=True, width="100%", height="1000px", bgcolor="#1a1a1a", font_color="#f0f0f0")

        if not self.priority_list or not self.betweenness:
            print("⚠️ Call analyze_impacts() first!")
            sampled_nodes = random.sample(list(self.graph.nodes()), min(num_critical_nodes, self.graph.number_of_nodes()))
            subgraph = self.graph.subgraph(sampled_nodes)
        else:
            top_critical_nodes = self.priority_list[:num_critical_nodes]
            neighbor_nodes = set(top_critical_nodes)
            for node in top_critical_nodes:
                neighbors = self.graph.neighbors(node)
                neighbor_nodes.update(neighbors)
            sampled_nodes = list(neighbor_nodes)
            subgraph = self.graph.subgraph(sampled_nodes)

        # Use short labels (just type) instead of full node names
        for node in subgraph.nodes():
            data = self.graph.nodes[node]
            node_type = data.get('type', 'unknown')
            label = node_type  # Just the type, e.g. 'hospital'
            degree = subgraph.degree(node)
            if node_type == 'hospital':
                color = '#ff6f69'
                size = degree * 3 + 15
            elif 'power' in node_type:
                color = '#4ecdc4'
                size = degree * 3 + 12
            elif node_type == 'water':
                color = '#a7dadc'
                size = degree * 2 + 10
            elif node_type == 'building':
                color = '#d9bf77'
                size = degree * 1.5 + 8
            else:
                color = '#f7fff7'
                size = degree * 1.5 + 8

            net.add_node(node, label=label, color=color, size=int(size))

        for u, v in subgraph.edges():
            edge_data = self.graph.get_edge_data(u, v)
            condition = edge_data.get('condition', 'good')
            color = '#cccccc' if condition == 'good' else '#ffaaaa'
            width = 1 if condition == 'good' else 2
            net.add_edge(u, v, weight=0.5, color=color, width=width)

        net.repulsion(node_distance=50, spring_length=100, spring_strength=0.01, central_gravity=0.2)
        net.show_buttons(filter_=['physics'])
        net.show(output_file)
        print(f"✅ Interactive graph saved as {output_file}")


def generate_pseudo_graph(num_nodes=100):
    G = nx.Graph()
    types = ['hospital', 'power_line', 'water', 'building']
    weights = [0.1, 0.3, 0.2, 0.4]

    for i in range(num_nodes):
        node_type = random.choices(types, weights)[0]
        G.add_node(f"Node_{i}", type=node_type)

    for i in range(num_nodes):
        connections = random.randint(1, 4)
        for _ in range(connections):
            target = random.randint(0, num_nodes - 1)
            if target != i:
                edge_type = random.choice(['road', 'power_line', 'water_pipe', 'fiber'])
                condition = random.choices(['good', 'bad'], weights=[0.8, 0.2])[0]
                G.add_edge(f"Node_{i}", f"Node_{target}", type=edge_type, condition=condition)

    return G


if __name__ == "__main__":
    print("🚀 Disaster Recovery System")

    pseudo_graph = generate_pseudo_graph(100)
    drs = DisasterRecoverySystem(pseudo_graph)

    drs.analyze_impacts()
    recs = drs.generate_recommendations()

    print("\nRecommended Actions:")
    print(recs.sort_values('urgency', ascending=False).to_string(index=False))

    #drs.visualize_interactive(num_critical_nodes=50, output_file="critical_network_100.html")


🚀 Disaster Recovery System

=== Critical Infrastructure Analysis ===

Top 5 Critical Nodes (Combined Score):
- Node_49 (hospital, Score: 0.54)
- Node_72 (power_line, Score: 0.46)
- Node_34 (hospital, Score: 0.38)
- Node_95 (power_line, Score: 0.34)
- Node_42 (water, Score: 0.33)

Worst-case Failure Impacts:
If Node_49 fails, it affects 40 nodes within 2 hops
If Node_72 fails, it affects 44 nodes within 2 hops
If Node_34 fails, it affects 20 nodes within 2 hops

=== Recovery Recommendations ===

Recommended Actions:
node_id       type                                               action  urgency
Node_42      water Standard protocol, ensure water trucks (Est. 10 hrs) 0.100920
Node_49   hospital                Priority crew allocation (Est. 8 hrs) 0.088947
Node_72 power_line  Immediate repair with backup generator (Est. 6 hrs) 0.041137
Node_95 power_line  Immediate repair with backup generator (Est. 6 hrs) 0.035857
Node_34   hospital                Priority crew allocation (Est. 8 hrs) 0.

In [ ]:
#####END OF PROJECT TILL PHASE - 2

In [4]:
# NOT PART OF PROJECT




import networkx as nx
import random
from pyvis.network import Network

def generate_pseudo_graph(num_nodes=100):
    G = nx.Graph()
    types = ['hospital', 'power_line', 'water', 'building']
    weights = [0.1, 0.3, 0.2, 0.4]  # probabilities for types

    # Add nodes with type attribute
    for i in range(num_nodes):
        node_type = random.choices(types, weights)[0]
        G.add_node(f"Node_{i}", type=node_type)

    # Add edges randomly
    for i in range(num_nodes):
        connections = random.randint(1, 4)
        for _ in range(connections):
            target = random.randint(0, num_nodes - 1)
            if target != i:
                edge_type = random.choice(['road', 'power_line', 'water_pipe', 'fiber'])
                condition = random.choices(['good', 'bad'], weights=[0.8, 0.2])[0]
                G.add_edge(f"Node_{i}", f"Node_{target}", type=edge_type, condition=condition)

    return G

def save_graph_as_html(G, filename="pseudo_graph_100.html"):
    net = Network(notebook=True, height="800px", width="100%", bgcolor="#222222", font_color="white")

    color_map = {
        'hospital': '#ff6f69',
        'power_line': '#4ecdc4',
        'water': '#a7dadc',
        'building': '#d9bf77',
        'unknown': '#f7fff7'
    }

    # Add actual graph nodes
    for node, data in G.nodes(data=True):
        node_type = data.get('type', 'unknown')
        color = color_map.get(node_type, '#f7fff7')
        net.add_node(node, label=node_type, color=color, title=node)

    # Add graph edges
    for u, v, data in G.edges(data=True):
        cond = data.get('condition', 'good')
        color = '#cccccc' if cond == 'good' else '#ff5555'
        width = 1 if cond == 'good' else 2
        net.add_edge(u, v, color=color, width=width)

    # --- Add Legend ---
    legend_x = -1500
    legend_y = -1000
    spacing_y = 100
    idx = 0

    # Add dummy legend nodes for node types
    for node_type, color in color_map.items():
        legend_node_id = f"legend_node_{node_type}"
        net.add_node(legend_node_id, label=node_type, color=color, x=legend_x, y=legend_y + spacing_y * idx,
                     physics=False, fixed=True)
        idx += 1

    # Add dummy edges for edge condition legend
    net.add_node("edge_legend_good", label="Edge: good", color="#cccccc", x=legend_x + 300, y=legend_y,
                 physics=False, fixed=True)
    net.add_node("edge_legend_bad", label="Edge: bad", color="#ff5555", x=legend_x + 300, y=legend_y + spacing_y,
                 physics=False, fixed=True)
    net.add_edge("edge_legend_good", "edge_legend_good", color="#cccccc", width=1)
    net.add_edge("edge_legend_bad", "edge_legend_bad", color="#ff5555", width=2)

    net.show_buttons(filter_=['physics'])
    net.show(filename)
    print(f"Graph with legend saved as {filename}")


if __name__ == "__main__":
    G = generate_pseudo_graph(100)
    save_graph_as_html(G, "pseudo_graph_100.html")


pseudo_graph_100.html
Graph with legend saved as pseudo_graph_100.html


In [3]:
!pip install pyvis

  Using cached pyvis-0.3.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jsonpickle-4.0.5-py3-none-any.whl.metadata (8.2 kB)
Using cached pyvis-0.3.2-py3-none-any.whl (756 kB)
Using cached jsonpickle-4.0.5-py3-none-any.whl (46 kB)

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
